# SC2001 Project 1 - Part (d): Hybrid vs original Merge Sort

> Implement the original merge sort and compare it against the hybrid on key
> comparisons and CPU time, on 10 million integers, using the optimal S from (c).

Algorithms from part (a) are imported unchanged from
`SC2001 Lab Section abc.i.py`; plain merge sort is implemented below.

**The result to look for:** the hybrid performs *more* key comparisons yet
finishes *faster*. Explaining that is the point of this part.

## Setup

In [ ]:
%matplotlib inline

import gc
import importlib.util
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

MODULE_PATH = Path("SC2001 Lab Section abc.i.py").resolve()
if not MODULE_PATH.exists():
    raise FileNotFoundError(f"Could not find {MODULE_PATH}. Run this notebook from the same folder as the .py file.")

spec = importlib.util.spec_from_file_location("hybrid_sort_lib", MODULE_PATH)
lib = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lib)

insertion_sort = lib.insertion_sort
merge          = lib.merge          # reused verbatim by plain merge sort below
hybrid_sort    = lib.hybrid_sort

print("Loaded algorithms from:", MODULE_PATH.name)

## Parameters

**Set `OPTIMAL_S` to the value your own (c.iii) run reported.** The valley there
is broad, so anything in roughly 8-32 gives the same conclusion.

`QUICK = True` for development. The full run sorts 10,000,000 integers several
times: ~8 minutes, peaking near 1 GB.

In [ ]:
QUICK = False              # True = fast development run, False = the real 10M run

SEED = 2001                # same seed scheme as part (c), so datasets match
MAX_VALUE_X = 1000000

OPTIMAL_S = 16             # <-- from part (c.iii); replace with your own result

if QUICK:
    N_HEADLINE     = 200000
    SIZES_SCALING  = [10000, 25000, 50000, 100000]     # kept below N_HEADLINE
    HEADLINE_REPEATS = 2
else:
    N_HEADLINE     = 10000000                 # the 10 million integers the question asks for
    SIZES_SCALING  = [100000, 250000, 500000, 1000000, 2500000, 5000000]
    HEADLINE_REPEATS = 3

print(f"QUICK = {QUICK}")
print(f"Head-to-head at n = {N_HEADLINE:,} with S = {OPTIMAL_S} (best of {HEADLINE_REPEATS} runs)")
print(f"Scaling study over {SIZES_SCALING}")

## Data and timing helpers

Same helpers as part (c), repeated so this notebook stands alone. `generate_one`
seeds from `n`, so a given size yields the identical array in both notebooks.

In [ ]:
def generate_one(n, x=MAX_VALUE_X, seed=SEED):
    """Generates one array of n random integers in [1, x], reproducibly."""
    random.seed(seed * 1000003 + n)
    return [random.randint(1, x) for _ in range(n)]


def time_sort(sort_fn, data, min_total=0.5, min_repeats=2, max_repeats=200):
    """Times sort_fn(array); returns (best_seconds, comparisons, runs).

    Repeats on a time budget, keeps the FASTEST run (the one least disturbed by
    other activity on the machine), disables the cyclic garbage collector so a
    collection pause cannot land on one algorithm and not the other, and makes
    the defensive copy outside the timed region. Reference counting still frees
    the temporaries, so nothing leaks while the collector is off.
    """
    best = float("inf")
    comparisons = None
    total = 0.0
    runs = 0
    gc_was_enabled = gc.isenabled()
    gc.disable()
    try:
        while runs < max_repeats and (runs < min_repeats or total < min_total):
            working = data[:]                  # not timed
            start = time.perf_counter()
            _, comparisons = sort_fn(working)
            elapsed = time.perf_counter() - start
            best = min(best, elapsed)
            total += elapsed
            runs += 1
    finally:
        if gc_was_enabled:
            gc.enable()
    return best, comparisons, runs

---
## The original merge sort

The lecture version: split, recurse, merge, bottoming out at a single element.

**It reuses `merge` from the `.py` file on purpose.** The only difference between
the two algorithms must be the base case; a separate merge routine would make the
measured gap part threshold and part implementation accident.

That makes plain merge sort *exactly* `hybrid_sort(lst, 1)` - with S=1 the
insertion sort only ever sees single-element subarrays. The cell below asserts
that equivalence, checking the baseline is faithful.

In [ ]:
def merge_sort(lst):
    """The original merge sort from lecture. Returns (sorted_list, comparisons)."""
    n = len(lst)

    if n <= 1:                       # base case: a single element is already sorted
        return lst[:], 0

    middle = n // 2

    left, left_comparison = merge_sort(lst[:middle])
    right, right_comparison = merge_sort(lst[middle:])
    sorted_list, merge_comparison = merge(left, right)

    comparison = left_comparison + right_comparison + merge_comparison
    return sorted_list, comparison


# --- verification of the baseline -------------------------------------------
_probe = generate_one(3000, 1000)

_ms_out, _ms_comp = merge_sort(_probe)
assert _ms_out == sorted(_probe), "merge_sort does not sort correctly"

_hy_out, _hy_comp = hybrid_sort(_probe, OPTIMAL_S)
assert _hy_out == sorted(_probe), "hybrid_sort does not sort correctly"

# plain merge sort must be indistinguishable from the hybrid with S = 1
_s1_out, _s1_comp = hybrid_sort(_probe, 1)
assert _ms_comp == _s1_comp, f"baseline mismatch: {_ms_comp} vs {_s1_comp}"

print(f"merge_sort and hybrid_sort(.., 1) agree exactly: {_ms_comp:,} comparisons on n=3,000")
print(f"hybrid_sort(.., S={OPTIMAL_S}) on the same array:      {_hy_comp:,} comparisons")
print("Both sort correctly. The baseline is faithful.")
del _probe, _ms_out, _hy_out, _s1_out

---
## Chart styling

Two choices that keep the figures honest: **comparisons and CPU time never share
a plot** (unrelated units - twin axes would invent a relationship), and **bars
start at zero** (the counts differ by a few percent; a truncated axis would make
that look like a chasm).

Blue is the hybrid, orange plain merge sort, in every figure.

In [ ]:
C_HYBRID = "#2a78d6"      # blue  - hybrid, in every figure
C_MERGE  = "#eb6834"      # orange - plain merge sort, in every figure
INK      = "#1a1a19"
MUTED    = "#6b6b66"
GRIDCLR  = "#e3e3de"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.edgecolor":   GRIDCLR,
    "axes.labelcolor":  MUTED,
    "text.color":       INK,
    "xtick.color":      MUTED,
    "ytick.color":      MUTED,
    "font.size":        10,
})


def style_axes(ax, grid_axis="y"):
    """Recessive chrome: hairline solid grid, no top/right spines."""
    ax.grid(True, axis=grid_axis, color=GRIDCLR, lw=0.8, ls="-")
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def human_time(seconds):
    """Formats a duration for labels and tables."""
    if seconds < 1:
        return f"{seconds * 1000:.0f} ms"
    return f"{seconds:.2f} s"


print("Chart styling ready.")

---
## The head-to-head on 10,000,000 integers

Both sort the identical array, timed with the same helper and repeat count.

In [ ]:
print(f"Generating {N_HEADLINE:,} random integers in [1, {MAX_VALUE_X:,}]...")
t0 = time.perf_counter()
data = generate_one(N_HEADLINE)
print(f"  done in {time.perf_counter() - t0:.1f}s\n")

print(f"Sorting with the HYBRID algorithm (S = {OPTIMAL_S})...")
hybrid_time, hybrid_comps, hybrid_runs = time_sort(
    lambda a: hybrid_sort(a, OPTIMAL_S), data, min_repeats=HEADLINE_REPEATS)
print(f"  {human_time(hybrid_time)}   {hybrid_comps:,} key comparisons   (best of {hybrid_runs})\n")

print("Sorting with the ORIGINAL merge sort...")
merge_time, merge_comps, merge_runs = time_sort(
    merge_sort, data, min_repeats=HEADLINE_REPEATS)
print(f"  {human_time(merge_time)}   {merge_comps:,} key comparisons   (best of {merge_runs})\n")

data = None      # release ~400 MB before plotting

comp_delta = (hybrid_comps - merge_comps) / merge_comps * 100
speedup    = merge_time / hybrid_time
time_saved = (merge_time - hybrid_time) / merge_time * 100

print("=" * 74)
print(f"{'':<26}{'Hybrid (S=' + str(OPTIMAL_S) + ')':>20}{'Merge sort':>20}")
print("-" * 74)
print(f"{'Key comparisons':<26}{hybrid_comps:>20,}{merge_comps:>20,}")
print(f"{'CPU time':<26}{human_time(hybrid_time):>20}{human_time(merge_time):>20}")
print("=" * 74)
print(f"Comparisons: the hybrid performs {abs(comp_delta):.2f}% "
      f"{'MORE' if comp_delta > 0 else 'FEWER'} than plain merge sort "
      f"({abs(hybrid_comps - merge_comps):,} {'extra' if comp_delta > 0 else 'saved'}).")
print(f"CPU time:    the hybrid is {speedup:.2f}x faster "
      f"({time_saved:.1f}% less time, {merge_time - hybrid_time:.2f}s saved).")

### Figure 1 - the two metrics disagree

On comparisons the bars are almost level; on time they are not.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
labels = [f"Hybrid\n(S = {OPTIMAL_S})", "Merge sort\n(original)"]
colors = [C_HYBRID, C_MERGE]

# Panel A: key comparisons, zero baseline so the near-tie reads as a near-tie
bars = ax1.bar(labels, [hybrid_comps, merge_comps], color=colors, width=0.42)
for bar, value in zip(bars, [hybrid_comps, merge_comps]):
    ax1.text(bar.get_x() + bar.get_width() / 2, value, f"{value / 1e6:,.3f}M",
             ha="center", va="bottom", fontsize=10, color=INK, fontweight="bold")
ax1.set_title("Key comparisons", color=INK, fontsize=11, pad=14, loc="left")
ax1.set_ylabel("Comparisons")
ax1.set_ylim(0, max(hybrid_comps, merge_comps) * 1.18)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda v, _: "0" if v == 0 else f"{v / 1e6:g}M"))
style_axes(ax1)

# Panel B: CPU time - a separate panel, never a second axis on panel A
bars = ax2.bar(labels, [hybrid_time, merge_time], color=colors, width=0.42)
for bar, value in zip(bars, [hybrid_time, merge_time]):
    ax2.text(bar.get_x() + bar.get_width() / 2, value, human_time(value),
             ha="center", va="bottom", fontsize=10, color=INK, fontweight="bold")
ax2.set_title("CPU time", color=INK, fontsize=11, pad=14, loc="left")
ax2.set_ylabel("Seconds")
ax2.set_ylim(0, max(hybrid_time, merge_time) * 1.18)
style_axes(ax2)

fig.suptitle(f"Hybrid vs original merge sort, n = {N_HEADLINE:,}",
             fontsize=13, color=INK, x=0.125, ha="left", y=1.0)
fig.text(0.125, 0.90,
         f"Hybrid does {abs(comp_delta):.1f}% more comparisons, yet runs {speedup:.2f}x faster",
         fontsize=10, color=MUTED, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

---
## Why more comparisons still means less time

The hybrid trades a few extra comparisons for a large cut in *structural* work.
Below the threshold, plain merge sort keeps recursing, and each call costs a
function invocation, two list slices, a fresh result list and the bookkeeping in
`merge`. The hybrid stops at size S and finishes in a tight loop, allocating
nothing.

Counting the calls makes the saving concrete, and needs no timing, so it is
exact and reproducible.

In [ ]:
def count_calls(n, s):
    """Number of recursive sort calls the hybrid makes on an array of size n.

    Mirrors the recursion in hybrid_sort exactly: a call returns immediately if
    n <= 1 or n <= s, otherwise it splits into floor(n/2) and ceil(n/2).

    Counted level by level rather than by actually recursing. Every node on a
    level has one of only two sizes (a parent of size m yields m//2 and m-m//2),
    so one dict per level is enough. Recursing here would mean 20 million Python
    calls at n = 10,000,000 - slower than the sorts being analysed.
    """
    if n <= 0:
        return 0
    level = {n: 1}
    total = 0
    while level:
        nxt = {}
        for size, count in level.items():
            total += count
            if size <= 1 or size <= s:
                continue                       # this call is a leaf: it does not recurse
            half = size // 2
            for child in (half, size - half):
                nxt[child] = nxt.get(child, 0) + count
        level = nxt
    return total


# A merge sort that bottoms out at size 1 builds a tree with n leaves and n-1
# internal nodes, so the count must be exactly 2n-1. Checks the counter itself.
assert count_calls(N_HEADLINE, 1) == 2 * N_HEADLINE - 1

calls_hybrid = count_calls(N_HEADLINE, OPTIMAL_S)
calls_merge  = count_calls(N_HEADLINE, 1)

print(f"Recursive calls on n = {N_HEADLINE:,}")
print(f"  original merge sort : {calls_merge:>15,}")
print(f"  hybrid (S = {OPTIMAL_S:<3})    : {calls_hybrid:>15,}")
print(f"  avoided by the hybrid: {calls_merge - calls_hybrid:>15,} "
      f"({(1 - calls_hybrid / calls_merge) * 100:.1f}% fewer)")
print()
print(f"Each avoided call also avoids two slices and one result-list allocation - against")
print(f"which the hybrid's {abs(hybrid_comps - merge_comps):,} extra comparisons "
      f"({abs(comp_delta):.2f}%) are cheap.")

---
## How the gap behaves as n grows

The 10M figure is one point; a range of sizes shows whether the advantage is
stable. The speedup ratio gets its own panel rather than a twin axis on the
timings.

In [ ]:
scaling = []
for n in SIZES_SCALING:
    d = generate_one(n)
    h_t, h_c, _ = time_sort(lambda a: hybrid_sort(a, OPTIMAL_S), d, min_repeats=1)
    m_t, m_c, _ = time_sort(merge_sort, d, min_repeats=1)
    d = None
    scaling.append({"n": n, "hybrid_time": h_t, "merge_time": m_t,
                    "hybrid_comps": h_c, "merge_comps": m_c})
    print(f"n = {n:>9,}   hybrid {human_time(h_t):>9}   merge {human_time(m_t):>9}   "
          f"speedup {m_t / h_t:.2f}x   comparisons {(h_c - m_c) / m_c * 100:+.2f}%")

ns          = [r["n"] for r in scaling]
hybrid_ts   = [r["hybrid_time"] for r in scaling]
merge_ts    = [r["merge_time"] for r in scaling]
speedups    = [r["merge_time"] / r["hybrid_time"] for r in scaling]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

ax1.plot(ns, merge_ts,  color=C_MERGE,  lw=2, marker="o", ms=8, label="Merge sort (original)")
ax1.plot(ns, hybrid_ts, color=C_HYBRID, lw=2, marker="o", ms=8, label=f"Hybrid (S = {OPTIMAL_S})")
ax1.set_xscale("log")
ax1.set_yscale("log")
# Identity is carried by the legend AND by a label at each line's end, so it is
# never colour-alone. Labels wear ink, not the series colour; the mark beside
# them carries the identity.
ax1.set_xlim(ns[0] * 0.8, ns[-1] * 2.6)
for label, series, colour in (("Merge sort", merge_ts, C_MERGE),
                              ("Hybrid", hybrid_ts, C_HYBRID)):
    ax1.annotate(label, xy=(ns[-1], series[-1]), xytext=(9, -3),
                 textcoords="offset points", fontsize=9, color=INK)
ax1.yaxis.set_major_locator(ticker.LogLocator(base=10, subs=(1.0, 2.0, 5.0), numticks=12))
ax1.yaxis.set_minor_locator(ticker.NullLocator())
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: human_time(v)))
ax1.set_title("CPU time vs input size", color=INK, fontsize=11, pad=14, loc="left")
ax1.set_xlabel("Input size n")
ax1.set_ylabel("CPU time")
ax1.legend(frameon=False, loc="upper left")
style_axes(ax1, grid_axis="both")

# Single series: the title names it, so no legend box
ax2.plot(ns, speedups, color=C_HYBRID, lw=2, marker="o", ms=8)
ax2.axhline(1.0, color=MUTED, lw=1, ls="-")
ax2.text(ns[0], 1.0, "  parity", va="bottom", ha="left", fontsize=9, color=MUTED)
ax2.annotate(f"{speedups[-1]:.2f}x", xy=(ns[-1], speedups[-1]),
             xytext=(-4, 8), textcoords="offset points",
             ha="right", fontsize=10, color=INK, fontweight="bold")
ax2.set_xscale("log")
ax2.set_ylim(0, max(speedups) * 1.25)
ax2.set_title("How many times faster the hybrid is", color=INK, fontsize=11, pad=14, loc="left")
ax2.set_xlabel("Input size n")
ax2.set_ylabel("Merge time / hybrid time")
style_axes(ax2)

fig.tight_layout()
plt.show()

---
## Results table

The numbers behind the figures, readable without relying on colour.

In [ ]:
print(f"{'n':>10}  {'hybrid time':>12}  {'merge time':>12}  {'speedup':>8}  "
      f"{'hybrid comps':>15}  {'merge comps':>15}  {'comp diff':>10}")
print("-" * 96)
for r in scaling:
    print(f"{r['n']:>10,}  {human_time(r['hybrid_time']):>12}  {human_time(r['merge_time']):>12}  "
          f"{r['merge_time'] / r['hybrid_time']:>7.2f}x  {r['hybrid_comps']:>15,}  "
          f"{r['merge_comps']:>15,}  {(r['hybrid_comps'] - r['merge_comps']) / r['merge_comps'] * 100:>+9.2f}%")
print("-" * 96)
print(f"{N_HEADLINE:>10,}  {human_time(hybrid_time):>12}  {human_time(merge_time):>12}  "
      f"{speedup:>7.2f}x  {hybrid_comps:>15,}  {merge_comps:>15,}  {comp_delta:>+9.2f}%   <-- part (d)")

---
## Conclusion

In [ ]:
print("PART (d) CONCLUSION")
print("=" * 78)
print(f"On {N_HEADLINE:,} random integers, with S = {OPTIMAL_S} from part (c.iii):")
print()
print(f"  Key comparisons   hybrid {hybrid_comps:,} vs merge sort {merge_comps:,}")
print(f"                    the hybrid performs {abs(comp_delta):.2f}% "
      f"{'more' if comp_delta > 0 else 'fewer'}")
print(f"  CPU time          hybrid {human_time(hybrid_time)} vs merge sort {human_time(merge_time)}")
print(f"                    the hybrid is {speedup:.2f}x faster")
print()
print("The two metrics disagree, and that is the substance of this part:")
print()
print("  * Both are Theta(n log n) in comparisons, so no large gap was available.")
print("    The hybrid gives a few percent away: insertion sort needs ~S^2/4 comparisons")
print("    per block where merge sort needs ~S log2(S).")
print()
print(f"  * In exchange, {calls_merge - calls_hybrid:,} recursive calls never happen "
      f"({(1 - calls_hybrid / calls_merge) * 100:.0f}% fewer),")
print("    nor their slices and allocations. That is where the time goes, and a")
print("    comparison counter cannot see any of it.")
print()
print("  * So: key comparisons for asymptotic analysis, but choosing S - or judging")
print("    whether the hybrid is worth using - requires measuring time.")
print("=" * 78)